# Singapore-2: proving termination with the DSL

A two-variable loop. Model it as a `dslModule`, load its ranking function `V(x, y)`, and
verify with Z3 that `V >= 0` and `V` strictly decreases on every iteration.

## The program

```c
int main() {
    int x = __VERIFIER_nondet_int();
    int y = __VERIFIER_nondet_int();

    // prevent overflows
    if (!(-65535 <= x && x <= 65535)) return 0;
    if (!(-65535 <= y && y <= 65535)) return 0;

    if (x + y <= 0) {
        while (x > 0) {
            x = x + x + y;
            y = y - 1;
        }
    }
    return 0;
}
```

We model the `while` loop as the transition system. Two parts of the program are handled
outside the module:

- the entry condition `if (x + y <= 0)` is a **precondition**; it is preserved by the body
  (`x' + y' = 2(x + y) - 1`), so it is a loop invariant, and we carry it in the verification
  **domain**;
- the overflow guards are **dropped** — over unbounded integers (`LIA`) they never fire, and
  termination over the integers is the stronger result.

In [1]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import z3

from zrth import LIA, Sort, Wire
from zrth.dsl import dslModule, nxt, ite
from zrth.torch import Module
from zrth import z3 as zz3

## The program as a `dslModule`

`x` and `y` are `(latched, next)` wire pairs with nondeterministic initial values. Both are
updated only while `x > 0`: `x' = x + x + y`, `y' = y - 1`. The `x > 0` guard is shared
between the two next-values.

In [2]:
class Singapore2(dslModule):
    def init(self, extl):
        x0, y0 = extl
        return nxt(x0), nxt(y0)

    def update(self, ctrl):
        x, y = ctrl
        loop = x > 0
        return ite(loop, x + x + y, x), ite(loop, y - 1, y)


INT = Sort.Int([1, 1])
x = (Wire(INT), Wire(INT))     # (latched, next)
y = (Wire(INT), Wire(INT))
x0 = (Wire(INT), Wire(INT))
y0 = (Wire(INT), Wire(INT))
program = Singapore2(theory=LIA, ctrl=(x, y), extl=(x0, y0))
print(program)

module
  external
    w4, w5 : Int(1, 1)
    w6, w7 : Int(1, 1)
  interface
    w0, w1 : Int(1, 1)
    w2, w3 : Int(1, 1)
  atom controls w1, w3 reads w0, w2 awaits w5, w7
  init
    Id w1; w5
    Id w3; w7
  update
    [[0]]
Tensor[[1, 1], Int64] w8; 
    Gt w9; w0, w8
    Add w10; w0, w0
    Add w11; w10, w2
    Ite w12; w9, w11, w0
    [[1]]
Tensor[[1, 1], Int64] w13; 
    Sub w14; w2, w13
    Ite w15; w9, w14, w2
    Id w1; w12
    Id w3; w15



`x` and `y` are controlled; `x0, y0` are external, so the module is open. `update` holds
both variables when `x <= 0` and applies the loop body otherwise.

## The ranking function

`V(x, y)` takes two inputs. Rebuild it from the JSON and wrap it with `zrth.torch.Module`.

In [3]:
# resolve the JSON whether cwd is this folder or the repo root
_name = "Singapore-2.ranking_function.json"
rf_path = next(d / _name for d in (Path.cwd(), Path.cwd() / "tutorials" / "singapore-2") if (d / _name).exists())
rf = json.loads(rf_path.read_text())
layers = rf["layers"]
delta = rf["delta_threshold"]


class Ranking(nn.Module):
    """Linear -> ReLU -> Linear, with shapes and weights from the JSON."""

    def __init__(self, layers):
        super().__init__()
        (o1, i1), (o2, i2) = [(len(L["W"]), len(L["W"][0])) for L in layers]
        self.fc1 = nn.Linear(i1, o1)
        self.fc2 = nn.Linear(i2, o2)
        with torch.no_grad():
            self.fc1.weight.copy_(torch.tensor(layers[0]["W"]))
            self.fc1.bias.copy_(torch.tensor(layers[0]["b"]))
            self.fc2.weight.copy_(torch.tensor(layers[1]["W"]))
            self.fc2.bias.copy_(torch.tensor(layers[1]["b"]))

    def forward(self, s):
        return self.fc2(torch.relu(self.fc1(s)))


V = Module(Ranking(layers))
print(V)

module
  external
    w16, w17 : Real(1, 2)
  interface
    w18, w19 : Real(1, 1)
  atom controls w19 awaits w17
  init
    Transpose w20; w17
    Linear w21; w20
    Transpose w22; w21
    ReLU w23; w22
    Transpose w24; w23
    Linear w25; w24
    Transpose w26; w25
    Id w19; w26
  update
    Transpose w20; w17
    Linear w21; w20
    Transpose w22; w21
    ReLU w23; w22
    Transpose w24; w23
    Linear w25; w24
    Transpose w26; w25
    Id w19; w26



## Verification

`x', y'` come from the program's `update`. `V` is evaluated on `(x, y)` and `(x', y')` (cast
to real). The domain is the loop condition `x >= 1` together with the invariant `x + y <= 0`.

In [4]:
(x_lat, x_nxt), (y_lat, y_nxt) = list(program.ctrl)
state = {x_lat: [z3.Int("x")], y_lat: [z3.Int("y")]}
for atom in program.atoms:
    for term in atom.update:
        state.update(zip(term.write, zz3.eval(term.itype, [state[w] for w in term.read])))

x_sym, y_sym = state[x_lat][0], state[y_lat][0]
print("x' =", state[x_nxt][0])
print("y' =", state[y_nxt][0])

extl, intf = list(V.extl), list(V.intf)

def eval_V(inp):
    r = {extl[0][1]: inp}
    for atom in V.atoms:
        for term in atom.update:
            r.update(zip(term.write, zz3.eval(term.itype, [r[w] for w in term.read])))
    return r[intf[0][1]][0]

V_s = eval_V([z3.ToReal(x_sym), z3.ToReal(y_sym)])
V_s_next = eval_V([z3.ToReal(state[x_nxt][0]), z3.ToReal(state[y_nxt][0])])
domain = [x_sym >= 1, x_sym + y_sym <= 0]

x' = If(0 < x, x + x + y, x)
y' = If(0 < x, y - 1, y)


**Condition 1** — `V(x, y) >= 0`.

In [5]:
solver = z3.Solver()
solver.add(*domain)
solver.add(V_s < 0)                        # negate
if solver.check() == z3.unsat:
    print("VERIFIED: V >= 0 on the domain")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: V >= 0 on the domain


**Condition 2** — `V(x, y) - V(x', y') >= delta`.

In [6]:
solver = z3.Solver()
solver.add(*domain)
solver.add(V_s - V_s_next < delta)         # negate
if solver.check() == z3.unsat:
    print(f"VERIFIED: V - V' >= {delta} at every step on the domain")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: V - V' >= 1.0 at every step on the domain


## Result

Both conditions hold on the domain, so `V` is a ranking function and the loop terminates.